In [120]:
import pandas as pd
import numpy as np
import altair as alt

# Using ecostyles package (installed in environment)
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name='article')

In [121]:
oecd2024 = pd.read_csv("oecdtaxdata2024_categories.csv")
oecdpanel = pd.read_csv("oecdtaxdata_panel.csv")

In [122]:
columnskeep = ['REF_AREA','Reference area','Institutional sector','OBS_VALUE']
oecd2024 = oecd2024[columnskeep]
oecd2024

,REF_AREA,Reference area,Institutional sector,OBS_VALUE
0,LVA,Latvia,Local government,2.424177
1,GRC,Greece,Local government,2.026000
2,FIN,Finland,Local government,13.973000
3,NLD,Netherlands,Local government,8.884000
4,LUX,Luxembourg,Local government,1.704149
...,...,...,...,...
173,EST,Estonia,Social security funds,2.233170
174,AUT,Austria,Social security funds,65.164100
175,IRL,Ireland,Local government,1.872131
176,LTU,Lithuania,Local government,0.274705


In [123]:
#computing the percentage of tax revenue collected by central government

# pivot: one row per country, sectors become columns
df_wide = oecd2024.pivot_table(
    index=['REF_AREA', 'Reference area'],
    columns='Institutional sector',
    values='OBS_VALUE',
    aggfunc='first'  # in case of duplicates, adjust if you have multiple years/periods
).reset_index()

# clean up column names (pivot_table can leave a MultiIndex/name on columns)
df_wide.columns.name = None

# replace missing government-sector values with 0 so ratios are computed cleanly
df_wide[['Central government', 'State government', 'Local government', 'European institutions and bodies (EU accounts)']] = (
    df_wide[['Central government', 'State government', 'Local government','European institutions and bodies (EU accounts)']]
    .fillna(0)
)

# compute ratio; zero values now mean missing sectors are treated as 0
df_wide['central_to_overall_ratio'] = (
    df_wide['Central government'] / (df_wide['General government'])
)

df_wide['central_to_overallno_SS_ratio'] = (
    df_wide['Central government'] / ((df_wide['General government']-df_wide['Social security funds']))
)

df_wide = df_wide[df_wide['Reference area'] != 'Japan'] # removing japan as there is no 2024 SS data

df_wide

,REF_AREA,Reference area,Central government,European institutions and bodies (EU accounts),General government,Local government,Social security funds,State government,central_to_overall_ratio,central_to_overallno_SS_ratio
0,AUT,Austria,137.790091,0.549032,214.307569,6.684977,65.164100,4.119368,0.642955,0.923876
1,BEL,Belgium,137.399300,1.799100,261.380700,11.979500,83.981000,26.221800,0.525667,0.774518
2,CAN,Canada,467.449885,0.000000,1072.389161,87.771701,112.067000,405.100576,0.435896,0.486764
3,CHE,Switzerland,75.885791,0.000000,224.033082,35.448921,55.043699,57.654672,0.338726,0.449057
4,CHL,Chile,55458.763870,0.000000,63820.279390,5341.986174,3019.529354,0.000000,0.868983,0.912139
5,COL,Colombia,251226.710200,0.000000,338777.365400,42944.030290,28943.903060,15662.721920,0.741569,0.810844
6,CRI,Costa Rica,7272.850825,0.000000,12200.437030,341.254400,4586.331800,0.000000,0.596114,0.955181
7,CZE,Czechia,1429.282278,10.882346,2737.378095,35.884463,1261.329008,0.000000,0.522135,0.968316
8,DEU,Germany,469.110000,5.583000,1645.939000,143.886000,643.305000,384.055000,0.285011,0.467878
9,DNK,Denmark,976.410964,3.694700,1322.277364,341.666700,0.505000,0.000000,0.738431,0.738713


In [124]:
bars = alt.Chart(df_wide).mark_bar().encode(
    x=alt.X(
        'Reference area:N',
        sort='-y',
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        'central_to_overall_ratio:Q',
        axis=alt.Axis(format='%', title=None)
    ),
    color=alt.condition(
        alt.datum.REF_AREA == 'GBR',
        alt.value('#d90000'),
        alt.value('#179fdb')
    ),
    tooltip=[alt.Tooltip('Reference area', title='Country'), alt.Tooltip('central_to_overall_ratio:Q', title='Central Government Revenue Share', format='.1%')]
).properties(
    width=800,
    height=350,
    title=alt.TitleParams(
        text='Central government share of total government tax revenue',
        subtitle='Data from OECD. Japan is omitted due to missing 2024 social security revenue data.',
        subtitleFontSize=12,
        anchor='start'
    )
)

chart = bars

chart

alt.Chart(...)

In [125]:
styles.save(chart, 'charts', 'oecdtaxcbarchart', width=600, height=200)

In [126]:
df_wide_stacked = df_wide.assign(
    central_share=df_wide["central_to_overall_ratio"],
    other_share=1 - df_wide["central_to_overall_ratio"]
)

df_stacked = df_wide_stacked.melt(
    id_vars=["REF_AREA", "Reference area", "central_to_overall_ratio"],
    value_vars=["central_share", "other_share"],
    var_name="share_type",
    value_name="share"
)

df_stacked["share_type"] = df_stacked["share_type"].replace({
    "central_share": "Central government",
    "other_share": "State, local, and other"
})

bars2 = alt.Chart(df_stacked).mark_bar().encode(
    x=alt.X(
        "Reference area:N",
        sort=alt.SortField("central_to_overall_ratio", order="descending"),
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "share:Q",
        axis=alt.Axis(format="%", title=None),
        stack="normalize"
    ),
    color=alt.Color(
        "share_type:N",
        scale=alt.Scale(
            domain=["State, local, and other", "Central government"],
            range=["#179fdb", "#143c54"]
        ),
        title=None,
        legend=alt.Legend(
            orient="bottom",
            direction="horizontal",
            labelLimit=200,
            symbolSize=100,
            symbolType="square"
        )
    ),
    opacity=alt.condition(
        alt.datum.REF_AREA == "GBR",
        alt.value(1),
        alt.value(0.8)
    ),
    tooltip=[
        alt.Tooltip("Reference area", title="Country"),
        alt.Tooltip("share_type:N", title="Component"),
        alt.Tooltip("share:Q", title="Share", format=".1%")
    ]
).properties(
    width=800,
    height=350,
    title=alt.TitleParams(
        text='Central government share of total government tax revenue',
        subtitle='Data from OECD. Japan is omitted due to missing 2024 social security revenue data.',
        subtitleFontSize=12,
        anchor='start'
    )
)

chart2 = bars2


In [127]:
styles.save(chart2, 'charts', 'oecdtaxcbarchart_cat', width=600, height=200)

In [130]:
chart2

alt.Chart(...)

In [128]:
df_wide_stacked = df_wide.assign(
    central_share=df_wide["central_to_overallno_SS_ratio"],
    other_share=1 - df_wide["central_to_overallno_SS_ratio"]
)

df_stacked = df_wide_stacked.melt(
    id_vars=["REF_AREA", "Reference area", "central_to_overallno_SS_ratio"],
    value_vars=["central_share", "other_share"],
    var_name="share_type",
    value_name="share"
)

df_stacked["share_type"] = df_stacked["share_type"].replace({
    "central_share": "Central government",
    "other_share": "State, local, and other"
})

bars3 = alt.Chart(df_stacked).mark_bar().encode(
    x=alt.X(
        "Reference area:N",
        sort=alt.SortField("central_to_overallno_SS_ratio", order="descending"),
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "share:Q",
        axis=alt.Axis(format="%", title=None),
        stack="normalize"
    ),
    color=alt.Color(
        "share_type:N",
        scale=alt.Scale(
            domain=["State, local, and other", "Central government"],
            range=["#179fdb", "#143c54"]
        ),
        title=None,
        legend=alt.Legend(
            orient="bottom",
            direction="horizontal",
            labelLimit=200,
            symbolSize=100,
            symbolType="square"
        )
    ),
    opacity=alt.condition(
        alt.datum.REF_AREA == "GBR",
        alt.value(1),
        alt.value(0.8)
    ),
    tooltip=[
        alt.Tooltip("Reference area", title="Country"),
        alt.Tooltip("share_type:N", title="Component"),
        alt.Tooltip("share:Q", title="Share", format=".1%")
    ]
).properties(
    width=800,
    height=350,
    title=alt.TitleParams(
        text='Central government share of total government tax revenue',
        subtitle='Data from OECD. Japan is omitted due to missing 2024 social security revenue data.',
        subtitleFontSize=12,
        anchor='start'
    )
)

chart3 = bars3


In [129]:
chart3

alt.Chart(...)